# Verified Grad-CAM region attribution on the patient-wise model

Trains one DenseNet + multi-scale attention model on the patient-wise split (DS1 train, DS2 test)
and computes Grad-CAM region attribution with a **corrected beat alignment**.

**Alignment.** Each beat is cut starting at its R-peak (index 0) and zero-padded at the end,
so the regions are defined in milliseconds *forward from the R-peak*, and the zero-padded tail
is excluded from the attribution:

| Region | ms from R-peak | samples (125 Hz) |
|---|---|---|
| QRS complex | 0 to 96 | 0 to 12 |
| ST segment | 96 to 224 | 12 to 28 |
| T-wave | 224 to 448 | 28 to 56 |
| Late diastole / next-beat onset | 448 to end of valid signal | 56 to valid length |

This replaces the earlier region map, which assumed the R-peak sat in the middle of the window.

**Before running:** GPU on, Internet on. About 10 to 15 minutes.

In [ ]:
!pip -q install wfdb

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import wfdb
from scipy.signal import resample_poly
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Conv1D, BatchNormalization, MaxPooling1D,
    Dense, Dropout, SpatialDropout1D, Flatten, Concatenate,
    GlobalAveragePooling1D, Add, MultiHeadAttention, LayerNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

SEED = int(os.environ.get('SEED', 0))

CLASS_NAMES = ['Normal', 'Supraventricular', 'Ventricular', 'Fusion']
NUM_CLASSES = 4
BEAT_LEN    = 187    # samples per beat at 125 Hz, same as the Kaggle files
TARGET_FS   = 125

# quick settings (leave as they are for the real run)
EPOCHS      = int(os.environ.get('EPOCHS', 50))
DATA_DIR    = os.environ.get('MITDB_DIR', 'mitdb')

print(f'TensorFlow {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 1. Beat extraction (identical to the main experiment)

In [ ]:
# DS1 / DS2 record lists from de Chazal et al. (2004). 22 recordings each.
# The paced recordings 102, 104, 107 and 217 are in neither list.
DS1 = [101, 106, 108, 109, 112, 114, 115, 116, 118, 119, 122, 124,
       201, 203, 205, 207, 208, 209, 215, 220, 223, 230]
DS2 = [100, 103, 105, 111, 113, 117, 121, 123, 200, 202, 210, 212,
       213, 214, 219, 221, 222, 228, 231, 232, 233, 234]
assert len(DS1) == 22 and len(DS2) == 22 and not set(DS1) & set(DS2)

# MIT-BIH beat symbols -> AAMI class index (N=0, S=1, V=2, F=3). Q beats are skipped.
AAMI = {}
for sym in ['N', 'L', 'R', 'e', 'j']:
    AAMI[sym] = 0
for sym in ['A', 'a', 'J', 'S']:
    AAMI[sym] = 1
for sym in ['V', 'E']:
    AAMI[sym] = 2
AAMI['F'] = 3
Q_SYMBOLS = ['/', 'f', 'Q']


def extract_beats(record_path):
    """Cut one record into beats. Returns (X, y, n_skipped_q).

    X has shape (n_beats, 187): each beat starts at its annotated R-peak, is
    1.2 x the median R-R interval long, is scaled to 0..1 and zero-padded.
    """
    record = wfdb.rdrecord(record_path, channel_names=['MLII'])
    ann = wfdb.rdann(record_path, 'atr')
    fs = int(record.fs)

    signal = resample_poly(record.p_signal[:, 0], TARGET_FS, fs)       # 360 Hz -> 125 Hz
    peaks = np.round(ann.sample * TARGET_FS / fs).astype(int)
    symbols = np.array(ann.symbol)

    is_beat = np.isin(symbols, list(AAMI) + Q_SYMBOLS)
    beat_peaks, beat_symbols = peaks[is_beat], symbols[is_beat]
    median_rr = np.median(np.diff(beat_peaks))
    window = int(min(BEAT_LEN, round(1.2 * median_rr)))

    X, y, skipped_q = [], [], 0
    for peak, sym in zip(beat_peaks, beat_symbols):
        if sym in Q_SYMBOLS:
            skipped_q += 1
            continue
        segment = signal[peak:peak + window]
        if len(segment) < window:
            continue                                   # beat cut off by the end of the record
        low, high = segment.min(), segment.max()
        if high - low < 1e-6:
            continue                                   # flat line, nothing to learn
        beat = np.zeros(BEAT_LEN, dtype=np.float32)
        beat[:window] = (segment - low) / (high - low)
        X.append(beat)
        y.append(AAMI[sym])
    return np.array(X, dtype=np.float32).reshape(-1, BEAT_LEN), np.array(y, dtype=int), skipped_q


def load_split(records, data_dir):
    X_all, y_all, groups, skipped = [], [], [], 0
    for rec in records:
        X, y, q = extract_beats(os.path.join(data_dir, str(rec)))
        X_all.append(X); y_all.append(y); groups.append(np.full(len(y), rec)); skipped += q
    return np.vstack(X_all), np.concatenate(y_all), np.concatenate(groups), skipped

In [ ]:
if not os.path.exists(os.path.join(DATA_DIR, '100.hea')):
    print('Downloading the MIT-BIH Arrhythmia Database from PhysioNet...')
    wfdb.dl_database('mitdb', dl_dir=DATA_DIR)

VAL_RECORDS = [106, 116, 124, 215, 223]
TRAIN_RECORDS = [r for r in DS1 if r not in VAL_RECORDS]

X_tr, y_tr, g_tr, _ = load_split(TRAIN_RECORDS, DATA_DIR)
X_va, y_va, g_va, _ = load_split(VAL_RECORDS, DATA_DIR)
X_te, y_te, g_te, _ = load_split(DS2, DATA_DIR)
assert not (set(g_tr) & set(g_va)) and not (set(g_tr) & set(g_te)) and not (set(g_va) & set(g_te))
print('train', X_tr.shape, 'val', X_va.shape, 'test', X_te.shape)

## 2. Augmentation, building blocks, model (identical to the main experiment)

In [ ]:
def augment_beat(signal, shift_range=10, scale_range=0.12, noise_std=0.01):
    """
    Apply three ECG-valid augmentations to a single 187-timestep beat.
    
    Args:
        signal      : 1D array of length 187
        shift_range : max time-shift in timesteps (±10 = ±80ms at 125Hz)
        scale_range : amplitude scaling ± this fraction
        noise_std   : standard deviation of additive Gaussian noise
    Returns:
        Augmented signal of same length 187
    """
    # 1. Random time-shift — roll the signal and zero-pad the wrapped edge
    shift = np.random.randint(-shift_range, shift_range)
    aug   = np.roll(signal, shift)
    if shift > 0:
        aug[:shift] = 0.0
    elif shift < 0:
        aug[shift:] = 0.0

    # 2. Random amplitude scaling (multiplicative, mean=1.0)
    scale = 1.0 + np.random.uniform(-scale_range, scale_range)
    aug   = aug * scale

    # 3. Additive Gaussian noise — simulates electrode noise
    aug = aug + np.random.normal(0, noise_std, aug.shape)

    # Clip to valid range [0, 1] (MIT-BIH signals are pre-normalised)
    return np.clip(aug, 0.0, 1.0)


def augment_minority_classes(X, y, target_count=5000, aug_classes=(1, 2, 3)):
    """
    Over-sample minority classes up to target_count by augmentation.
    
    Args:
        X             : raw signals, shape (N, 187)
        y             : integer labels, shape (N,)
        target_count  : desired minimum samples per minority class
        aug_classes   : which class indices to augment (S=1, V=2, F=3)
    Returns:
        X_aug, y_aug  : augmented dataset (original + synthetic samples)
    """
    X_extra, y_extra = [], []
    for cls in aug_classes:
        mask      = (y == cls)
        X_cls     = X[mask]           # all real samples for this class
        n_real    = len(X_cls)
        n_needed  = max(0, target_count - n_real)
        if n_needed == 0:
            continue
        # Randomly pick which real samples to augment (with replacement)
        idxs = np.random.choice(n_real, n_needed, replace=True)
        for idx in idxs:
            X_extra.append(augment_beat(X_cls[idx]))
            y_extra.append(cls)

    if X_extra:
        X_aug = np.vstack([X, np.array(X_extra, dtype=np.float32)])
        y_aug = np.concatenate([y, np.array(y_extra, dtype=int)])
    else:
        X_aug, y_aug = X, y
    return X_aug, y_aug

In [ ]:
# ──────────────────────────────────────────────────────────────
# SHARED BUILDING BLOCKS & EVALUATION UTILITIES
# ──────────────────────────────────────────────────────────────

def dense_block(x, filters):
    """
    DenseNet block: each layer receives all previous feature maps.
    Output = concatenation of [input, conv1_out, conv2_out].
    This dense connectivity provides two benefits:
      - Feature reuse: early low-level ECG features survive to later layers
      - Implicit deep supervision: gradients flow directly to early layers
    """
    c1 = Conv1D(filters, 3, padding='same', activation='relu')(x)
    c1 = BatchNormalization()(c1)
    c1 = SpatialDropout1D(0.1)(c1)   # Drops entire feature maps, more effective than Dropout
    m1 = Concatenate()([x, c1])

    c2 = Conv1D(filters, 3, padding='same', activation='relu')(m1)
    c2 = BatchNormalization()(c2)
    c2 = SpatialDropout1D(0.1)(c2)
    return Concatenate()([x, c1, c2])


def transition_layer(x, out_filters):
    """
    Transition layer between dense blocks.
    Problem it solves: dense_block output has x + c1 + c2 channels.
    After 3 blocks (32→64→128), you'd have ~500 channels WITHOUT this.
    The 1×1 conv compresses back to out_filters (halving), keeping
    the model lean and reducing overfitting risk.
    This is the standard DenseNet-BC design (Huang et al., 2017).
    """
    x = Conv1D(out_filters, 1, padding='same', activation='relu')(x)  # 1×1 compression
    x = BatchNormalization()(x)
    return MaxPooling1D(2)(x)  # Halve temporal resolution


def attention_block(x, num_heads=4, key_dim=16, name_prefix='attn'):
    """
    Multi-Head Self-Attention block with residual connection.
    
    How it helps ECG classification:
    - Attention heads learn to focus on different ECG regions simultaneously
      (one head for QRS complex, another for T-wave, etc.)
    - The residual Add() preserves original features so the block
      is additive — it can only enhance, never destroy, existing features
    - LayerNorm stabilises training by normalising across feature dimension
    """
    # Project to attention dimension first (reduces computation)
    proj = Conv1D(num_heads * key_dim, 1, padding='same', activation='relu',
                  name=f'{name_prefix}_proj')(x)

    # Self-attention: Q=K=V=proj (the signal attends to itself)
    attn = MultiHeadAttention(
        num_heads=num_heads, key_dim=key_dim,
        dropout=0.1, name=f'{name_prefix}_mha'
    )(proj, proj)

    # Project back to original channel size for the residual addition
    attn = Conv1D(x.shape[-1], 1, padding='same', name=f'{name_prefix}_back')(attn)

    # Residual connection + LayerNormalization
    x = Add(name=f'{name_prefix}_add')([x, attn])
    return LayerNormalization(name=f'{name_prefix}_ln')(x)

In [ ]:
def build_proposed_model():
    inp = Input(shape=(187, 1), name='ecg_input')

    # ── Stage 1: Low-level feature extraction ─────────────────
    # Broad kernel (5) captures slow waveform components (ST segment and T-wave)
    x = Conv1D(32, 5, padding='same', activation='relu')(inp)
    x = BatchNormalization()(x)
    x = dense_block(x, 32)
    x = transition_layer(x, out_filters=32)   # 187 → 93 timesteps

    # ── Stage 2: Mid-level feature extraction ─────────────────
    x = Conv1D(64, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = dense_block(x, 64)
    x = transition_layer(x, out_filters=64)   # 93 → 46 timesteps

    # ── ATTENTION BLOCK 1: Mid-scale (46 timesteps ≈ 128ms) ───
    # At this resolution, the model can learn which temporal segments
    # (PR interval, QRS onset, ST segment) are most discriminative
    # for each arrhythmia class.
    # 4 heads × key_dim 16 = 64 attention dimensions
    x = attention_block(x, num_heads=4, key_dim=16, name_prefix='attn1')

    # ── Stage 3: High-level feature extraction ────────────────
    x = Conv1D(128, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x)
    x = dense_block(x, 128)
    x = transition_layer(x, out_filters=128)  # 46 → 23 timesteps

    # ── ATTENTION BLOCK 2: Fine-scale (23 timesteps ≈ 64ms) ───
    # At this compressed resolution, attention learns which of the
    # high-level features (e.g., QRS notch, ST elevation) are
    # diagnostically critical for the final classification.
    # 8 heads for richer attention diversity at the decision layer.
    x = attention_block(x, num_heads=8, key_dim=16, name_prefix='attn2')

    # Global average pooling → 128-dim beat representation
    x = GlobalAveragePooling1D()(x)

    # ── Classification head ───────────────────────────────────
    x = Dense(128, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = Dropout(0.4)(x)
    x = Dense(64, activation='relu',
              kernel_regularizer=tf.keras.regularizers.l2(1e-4))(x)
    x = Dropout(0.3)(x)
    out = Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)

    return Model(inp, out, name='DenseNet_MultiScale_Attention')

## 3. Train the DenseNet + attention model, patient-wise, one seed

In [ ]:
tf.keras.backend.clear_session()
tf.keras.utils.set_random_seed(SEED)

X_aug, y_aug = augment_minority_classes(X_tr, y_tr, target_count=5000)
weights = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=y_aug)
X_fit, y_fit = shuffle(X_aug.reshape(-1, BEAT_LEN, 1),
                       tf.keras.utils.to_categorical(y_aug, NUM_CLASSES), random_state=SEED)

model = build_proposed_model()
model.compile(optimizer=Adam(learning_rate=5e-4, clipnorm=1.0),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(X_fit, y_fit, epochs=EPOCHS, batch_size=64, verbose=2,
          validation_data=(X_va.reshape(-1, BEAT_LEN, 1), tf.keras.utils.to_categorical(y_va, NUM_CLASSES)),
          class_weight=dict(enumerate(weights)),
          callbacks=[EarlyStopping(monitor='val_accuracy', patience=7, restore_best_weights=True),
                     ReduceLROnPlateau(monitor='val_accuracy', factor=0.5, patience=3, min_lr=1e-7)])

y_pred = np.argmax(model.predict(X_te.reshape(-1, BEAT_LEN, 1), verbose=0), axis=1)
print(classification_report(y_te, y_pred, target_names=CLASS_NAMES, digits=3, zero_division=0))

## 4. Grad-CAM region attribution (corrected alignment)

Target layer: the output feeding the global pooling (attention block 2, 23 timesteps).
The heatmap is upsampled to 187 samples, averaged over correctly classified test beats of each
class, and integrated over the regions above, with the zero-padded tail of each beat excluded.

In [ ]:
# feature map feeding the global pooling: output of attention block 2 (23 timesteps)
target_layer = model.get_layer('attn2_ln')
print('Grad-CAM target layer:', target_layer.name, 'output', target_layer.output.shape)

grad_model = Model(model.inputs, [target_layer.output, model.output])

def gradcam_batch(X, class_idx):
    """Mean Grad-CAM heatmap (length 187) over the beats in X for class_idx."""
    Xt = tf.convert_to_tensor(X.reshape(-1, BEAT_LEN, 1), dtype=tf.float32)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(Xt, training=False)
        loss = preds[:, class_idx]
    grads = tape.gradient(loss, conv_out)                 # (n, t, c)
    weights = tf.reduce_mean(grads, axis=1)               # (n, c)
    cam = tf.reduce_sum(conv_out * weights[:, None, :], axis=-1)   # (n, t)
    cam = tf.nn.relu(cam).numpy()
    # upsample each cam from t to 187
    t = cam.shape[1]
    xp = np.linspace(0, 1, t); xq = np.linspace(0, 1, BEAT_LEN)
    up = np.stack([np.interp(xq, xp, c) for c in cam])
    return up                                             # (n, 187)

# correctly classified test beats per class
correct = (y_pred == y_te)
REGIONS = [('QRS', 0, 12), ('ST', 12, 28), ('T-wave', 28, 56), ('Late/TP', 56, BEAT_LEN)]
rows = []
heatmaps = {}
for cls in range(NUM_CLASSES):
    idx = np.where(correct & (y_te == cls))[0]
    if len(idx) == 0:
        continue
    Xc = X_te[idx]
    hm = gradcam_batch(Xc, cls)                           # (n,187)
    valid = (Xc > 1e-6)                                   # exclude zero padding
    hm = hm * valid
    mean_hm = hm.mean(axis=0)
    heatmaps[cls] = (Xc.mean(axis=0), mean_hm, len(idx))
    region_vals = []
    for name, a, b in REGIONS:
        region_vals.append(mean_hm[a:b].sum())
    region_vals = np.array(region_vals)
    region_pct = 100 * region_vals / region_vals.sum() if region_vals.sum() > 0 else region_vals
    rows.append([CLASS_NAMES[cls], len(idx)] + list(np.round(region_pct, 1)))

region_df = pd.DataFrame(rows, columns=['Class', 'n_beats'] + [r[0] for r in REGIONS])
region_df.to_csv('gradcam_regions_patientwise.csv', index=False)
print(region_df.to_string(index=False))

In [ ]:
# overlay figure: mean beat with Grad-CAM heatmap per class
fig, axes = plt.subplots(1, len([c for c in heatmaps]), figsize=(4*len(heatmaps), 3.2))
if len(heatmaps) == 1: axes = [axes]
for ax, cls in zip(axes, sorted(heatmaps)):
    beat, hm, n = heatmaps[cls]
    valid_len = int(np.max(np.where(beat > 1e-6)[0])) + 1 if np.any(beat > 1e-6) else BEAT_LEN
    t = np.arange(valid_len)
    ax.plot(t, beat[:valid_len], color='#222', lw=1.2)
    hmn = hm[:valid_len] / (hm[:valid_len].max() + 1e-9)
    ax.fill_between(t, 0, beat[:valid_len], where=None, alpha=0.0)
    sc = ax.scatter(t, beat[:valid_len], c=hmn, cmap='Reds', s=10, vmin=0, vmax=1)
    for _, a, b in REGIONS[:3]:
        ax.axvline(b, color='#999', ls=':', lw=0.7)
    ax.set_title(f'{CLASS_NAMES[cls]} (n={n})', fontsize=10)
    ax.set_xlabel('samples from R-peak'); ax.set_yticks([])
plt.tight_layout()
plt.savefig('gradcam_overlays_patientwise.png', dpi=160, bbox_inches='tight')
plt.show()
print('saved gradcam_regions_patientwise.csv and gradcam_overlays_patientwise.png')